# T2.3 Unit Mapping & T2.5 Data Loading

This notebook covers two tasks for the NO2 Air Quality Classification project:
- **T2.3**: Push unit-of-measurement ontology mappings to DBRepo column metadata
- **T2.5**: Load the raw EEA Parquet files into DBRepo and verify the SQL views

**Setup:**
- `.env` file must exist in the repo root with `TU_USERNAME` and `TU_PASSWORD`
- Database: `ed890fa1-154c-4a66-8529-4088c97f68db` on `https://test.dbrepo.tuwien.ac.at`
- Run cells top to bottom in order

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'dbrepo', 'pandas', 'pyarrow', 'python-dotenv', '-q'])

In [ ]:
import os
import decimal
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient

load_dotenv(dotenv_path=Path('../.env'))
TU_USERNAME = os.getenv('TU_USERNAME')
TU_PASSWORD = os.getenv('TU_PASSWORD')

DATABASE_ID = "ed890fa1-154c-4a66-8529-4088c97f68db"
DBREPO_ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
RAW_DATA_DIR = Path("../data/raw")

# hardcoded table IDs from the confirmed DBRepo schema
TABLE_IDS = {
    "measurements":       "0082a413-69b5-40a3-ac29-a7243961312d",
    "observation_logs":   "c51d477c-db84-4fae-b03c-3e7fb9c9be61",
    "verification_flags": "2e096722-778b-4f12-a236-5f32ad9556e4",
    "validity_flags":     "4b62c1b0-295b-4697-8c91-f61f075c8348",
    "aggregation_types":  "868cdaaa-c961-4833-af05-6b9cc5f7c904",
    "measurement_units":  "b1afdf57-ff37-4452-94e3-77c28bafc4b3",
    "pollutants":         "13dec6c5-5c0d-4a98-9973-b862c10b4e49",
    "sampling_points":    "b277e2ed-1fab-4ac3-962b-b1e82026ae5e"
}

client = RestClient(
    endpoint=DBREPO_ENDPOINT,
    username=TU_USERNAME,
    password=TU_PASSWORD
)

print(f"Username : {TU_USERNAME}")
print(f"Database : {DATABASE_ID}")
print(f"Parquet files: {len(list(RAW_DATA_DIR.glob('*.parquet')))}")
print("Client ready!")

## Verify Database

In [ ]:
try:
    db = client.get_database(database_id=DATABASE_ID)
    print(f"Database : {db.name}")
    print(f"Tables   : {[t.name for t in db.tables]}")
    print(f"Public   : {db.is_public}")
except Exception as e:
    print(f"Error: {e}")

## T2.3 - Unit Mappings

Mapping numeric columns to unit ontology concepts.

- **SI Digital Framework** is used for timestamp columns (second is an SI base unit)
- **QUDT** is used as the specified fallback for µg/m³ (prefixed derived unit not in SI Framework TTL)
- **QUDT** is also used for the dimensionless `data_capture` fraction

In [ ]:
UNIT_MAPPINGS = [
    {
        "column": "value",
        "table_id": TABLE_IDS["measurements"],
        "column_id": "6586931b-2ba1-4baa-93a6-0bb268e71bb0",
        "unit_symbol": "ug.m-3",
        "unit_uri": "http://qudt.org/vocab/unit/MicroGM-PER-M3",
        "ontology": "QUDT"
    },
    {
        "column": "data_capture",
        "table_id": TABLE_IDS["measurements"],
        "column_id": "a33142df-e598-4df9-a984-2e1927dac1f1",
        "unit_symbol": "1",
        "unit_uri": "http://qudt.org/vocab/unit/FRACTION",
        "ontology": "QUDT"
    },
    {
        "column": "start_time",
        "table_id": TABLE_IDS["measurements"],
        "column_id": "5a3b96b0-7cec-4266-80eb-77cebd9f5446",
        "unit_symbol": "s",
        "unit_uri": "http://si-digital-framework.org/SI/units/S",
        "ontology": "SI Digital Framework"
    },
    {
        "column": "end_time",
        "table_id": TABLE_IDS["measurements"],
        "column_id": "abf66b59-2a29-4922-8a9c-f6f0d53b10c5",
        "unit_symbol": "s",
        "unit_uri": "http://si-digital-framework.org/SI/units/S",
        "ontology": "SI Digital Framework"
    },
    {
        "column": "result_time",
        "table_id": TABLE_IDS["measurements"],
        "column_id": "5b0f69c8-1db3-4ab0-8160-13bd28a1a6e0",
        "unit_symbol": "s",
        "unit_uri": "http://si-digital-framework.org/SI/units/S",
        "ontology": "SI Digital Framework"
    }
]

print(f"Unit mappings to apply: {len(UNIT_MAPPINGS)}")
for m in UNIT_MAPPINGS:
    print(f"  measurements.{m['column']} -> {m['unit_symbol']} ({m['ontology']})")

In [ ]:
ok, fail = 0, 0
for m in UNIT_MAPPINGS:
    try:
        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=m["table_id"],
            column_id=m["column_id"],
            unit_uri=m["unit_uri"]
        )
        print(f"  + measurements.{m['column']} -> {m['unit_symbol']}")
        ok += 1
    except Exception as e:
        print(f"  x measurements.{m['column']}: {e}")
        fail += 1

print(f"\nResult: {ok} applied, {fail} failed")

## T2.5 - Load Data into DBRepo

The DBRepo import endpoint returned intermittent HTTP 503 errors during direct notebook uploads. To keep the workflow reproducible, this notebook now delegates loading to `scripts/load_dbrepo_import_csvs.py`.

The loader uses the prepared CSV files in `outputs/dbrepo_import/`, imports lookup tables before measurements, uploads in smaller chunks, and skips rows that already exist in DBRepo. This makes repeated runs safe and avoids duplicate primary keys.


In [ ]:
parquet_files = sorted(RAW_DATA_DIR.glob("*.parquet"))
dfs = []
for f in parquet_files:
    df = pd.read_parquet(f)
    df["source_file"] = f.stem
    dfs.append(df)

all_data = pd.concat(dfs, ignore_index=True)

# fix Decimal type to avoid JSON serialization errors
for col in all_data.columns:
    if all_data[col].dtype == object:
        all_data[col] = all_data[col].apply(
            lambda x: float(x) if isinstance(x, decimal.Decimal) else x
        )

print(f"Total rows   : {len(all_data):,}")
print(f"Columns      : {all_data.columns.tolist()}")

In [ ]:
from pathlib import Path
import subprocess
import sys

candidate_paths = [
    Path.cwd() / "scripts" / "load_dbrepo_import_csvs.py",
    Path.cwd().parent / "scripts" / "load_dbrepo_import_csvs.py",
]
script_path = next((p for p in candidate_paths if p.exists()), None)
if script_path is None:
    raise FileNotFoundError("Could not find scripts/load_dbrepo_import_csvs.py")

result = subprocess.run(
    [sys.executable, str(script_path)],
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout)


In [ ]:
expected_counts = {
    "sampling_points": 16,
    "pollutants": 1,
    "measurement_units": 1,
    "aggregation_types": 1,
    "validity_flags": 2,
    "verification_flags": 1,
    "observation_logs": 19,
    "measurements": 140160,
}

for name, expected in expected_counts.items():
    df = client.get_table_data(
        database_id=DATABASE_ID,
        table_id=TABLE_IDS[name],
        size=max(expected, 10),
    )
    actual = len(df)
    status = "OK" if actual == expected else "CHECK"
    print(f"{name}: {actual:,} rows ({status}; expected {expected:,})")


In [ ]:
print("Measurement preparation is handled by outputs/dbrepo_import/*.csv and scripts/load_dbrepo_import_csvs.py.")


In [ ]:
print("Sample import skipped: the idempotent loader already inserted the DBRepo rows and avoids duplicates.")


In [ ]:
print("Full import skipped here. Use scripts/load_dbrepo_import_csvs.py for repeatable chunked loading.")


## T2.5 - Verify SQL Views

In [ ]:
db = client.get_database(database_id=DATABASE_ID)
print(f"Views found: {len(db.views)}")
for view in db.views:
    try:
        df = client.get_view_data(database_id=DATABASE_ID, view_name=view.name, df=True)
        print(f"  + {view.name}: {len(df)} rows")
    except Exception as e:
        print(f"  x {view.name}: {e}")

## Summary

| Task | Description |
|------|-------------|
| T2.3 | Unit mappings pushed to DBRepo column metadata |
| T2.5 | Lookup tables and measurements loaded into DBRepo with the idempotent CSV loader |
| T2.5 | DBRepo row counts verified |
| T2.5 | SQL views can be verified after view creation is complete |

### Unit Mapping Summary

| Table | Column | Unit | Ontology | URI |
|-------|--------|------|----------|-----|
| measurements | value | microgram per cubic metre | QUDT | http://qudt.org/vocab/unit/MicroGM-PER-M3 |
| measurements | data_capture | 1 (fraction) | QUDT | http://qudt.org/vocab/unit/FRACTION |
| measurements | start_time | second | SI Digital Framework | http://si-digital-framework.org/SI/units/S |
| measurements | end_time | second | SI Digital Framework | http://si-digital-framework.org/SI/units/S |
| measurements | result_time | second | SI Digital Framework | http://si-digital-framework.org/SI/units/S |

### DBRepo Loading Summary

The DBRepo import is handled by `scripts/load_dbrepo_import_csvs.py` using CSV exports from `outputs/dbrepo_import/`. The script is safe to re-run because it checks existing primary keys and imports only missing rows.

Expected loaded row counts:

| Table | Rows |
|-------|------|
| sampling_points | 16 |
| pollutants | 1 |
| measurement_units | 1 |
| aggregation_types | 1 |
| validity_flags | 2 |
| verification_flags | 1 |
| observation_logs | 19 |
| measurements | 140,160 |
